# TP 2 — Premiers pas PySpark

**Big Data Engineering — Master 1 — DMI/FST/UCAD — Prof. Samba Ndiaye**

Ce notebook guide les parties B (WordCount), C (fil rouge) et D (Spark UI)
du TP 2. Complétez les cellules marquées `À COMPLÉTER`, exécutez tout de bout
en bout, puis poussez le notebook **avec ses sorties** dans `notebooks/`.

**Livrable** : ce notebook, avec le WordCount **commenté ligne à ligne**, les
observations de la Spark UI, et la comparaison Spark vs Pandas.

> Réflexe d'ingénieur : on ne dit pas « c'est lent », on dit « 4,2 s pour
> 50 000 lignes ». Mesurez tout.

## 0. Vérification de l'environnement

PySpark s'exécute sur la JVM : un **JDK 17** est requis. Si la cellule
échoue, revenez à la partie A du TP (`java -version`).

In [ ]:
import sys, platform
import pyspark
print("Python  :", sys.version.split()[0], "-", platform.system())
print("PySpark :", pyspark.__version__)

## 1. Créer la SparkSession

`master("local[*]")` exécute Spark dans ce notebook, sur **tous les cœurs**
de la machine. La Spark UI démarre sur http://localhost:4040 — ouvrez-la.

In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("TP2-WordCount")
         .master("local[*]")
         .getOrCreate())

print("Spark", spark.version)
print("Coeurs vus :", spark.sparkContext.defaultParallelism)
print("Spark UI   :", spark.sparkContext.uiWebUrl)

Spark 3.5.1
Coeurs vus : 12
Spark UI   : http://10.2.222.126:4040


## Exercice B — WordCount

Préparez d'abord un fichier texte dans `data/discours.txt` (un discours, un
article — au moins quelques centaines de mots). La cellule ci-dessous compte
les mots. **Commentez chaque ligne** : c'est le cœur du livrable.

In [ ]:
# === À COMPLÉTER ===

# Importe les fonctions Spark nécessaires au traitement des mots
from pyspark.sql.functions import explode, split, lower, col

# Lit le fichier texte et le charge dans un DataFrame Spark
texte = spark.read.text("../data/discours.txt")

mots = (
    texte

    # lower() met le texte en minuscules.
    # split() découpe chaque ligne en mots en utilisant les espaces.
    # explode() transforme la liste de mots obtenue en une ligne par mot.
    .select(
        explode(
            split(lower(col("value")), r"\s+")
        ).alias("mot")
    )

    # Supprime les éventuelles chaînes de caractères vides
    .filter(col("mot") != "")

    # Regroupe les mots identiques.
    # groupBy provoque un shuffle car Spark doit redistribuer les données
    # entre les partitions pour réunir les mêmes mots.
    .groupBy("mot")

    # Compte le nombre d'occurrences de chaque mot
    .count()
)

# Trie les mots du plus fréquent au moins fréquent
# et affiche les 15 premiers
mots.orderBy(col("count").desc()).show(15)


+----------+-----+
|       mot|count|
+----------+-----+
|       les|    5|
|       des|    5|
|        de|    4|
|     spark|    4|
|        et|    4|
|      être|    3|
|      pour|    3|
|   données|    3|
|        le|    3|
|      avec|    3|
|importants|    2|
|       qui|    2|
| plusieurs|    2|
|   traités|    2|
|   volumes|    2|
+----------+-----+
only showing top 15 rows



In [9]:
mots.orderBy(col("count").desc()).show(15)

+----------+-----+
|       mot|count|
+----------+-----+
|       les|    5|
|       des|    5|
|        de|    4|
|     spark|    4|
|        et|    4|
|      être|    3|
|      pour|    3|
|   données|    3|
|        le|    3|
|      avec|    3|
|importants|    2|
|       qui|    2|
| plusieurs|    2|
|   traités|    2|
|   volumes|    2|
+----------+-----+
only showing top 15 rows



### B — Observer (répondez en markdown)

1. Quelles lignes sont des **transformations** ? 
.select(...)
.filter(...)
.groupBy(...)
.count()
.orderBy(...)
   Laquelle est l'**action** ?
   .show(15)
2. Où se situe le **shuffle** ?
   .groupBy("mot")
3. Si vous relancez `mots.orderBy(...).show()`, pourquoi tout est-il
   recalculé ?
   Parce que Spark fonctionne de manière paresseuse et ne garde pas automatiquement le résultat en mémoire.Donc, à chaque nouvel appel de : mots.orderBy(col("count").desc()).show(15).Spark refait le calcul.

*Vos réponses :* …

## Exercice C — Recharger le fil rouge avec Spark

On reprend les fichiers du fil rouge, cette fois avec Spark. `inferSchema`
demande à Spark de **deviner** les types (pratique mais coûteux : une passe
de lecture en plus).

In [ ]:


# Lecture du fichier CSV avec la première ligne comme en-tête
# et détection automatique des types des colonnes
orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../data/orders.csv")
)

# Lecture du fichier JSON Lines
events = spark.read.json("../data/events.json")

# Affiche la structure des colonnes de orders
orders.printSchema()

# Compte et affiche le nombre de lignes de orders
print("orders :", orders.count())

# Affiche la structure des colonnes de events
events.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- date_commande: timestamp (nullable = true)
 |-- statut: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- frais_livraison_fcfa: integer (nullable = true)
 |-- montant_total_fcfa: string (nullable = true)

orders : 50000
root
 |-- device: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- ville: string (nullable = true)



### C — Explorer sans tout rapatrier

`show(n)` et `count()` sont des actions ; `select`, `filter`, `groupBy`
décrivent seulement le plan. **Ne jamais** faire `events.collect()`.

In [14]:


# Affiche 5 lignes avec seulement les colonnes choisies
orders.select("order_id", "statut", "canal").show(5)

# Filtre uniquement les commandes livrées
livrees = orders.filter(col("statut") == "livrée")

# Compte le nombre de commandes livrées
print("livrees :", livrees.count())

# Compte le nombre de commandes par statut
orders.groupBy("statut").count().show()

# Compte le nombre de commandes par canal
# puis trie du plus grand nombre au plus petit
orders.groupBy("canal") \
      .count() \
      .orderBy(col("count").desc()) \
      .show()
...

+--------+-------+----------+
|order_id| statut|     canal|
+--------+-------+----------+
|O0000001| livrée|mobile_app|
|O0000002| livrée|mobile_app|
|O0000003| livrée|       web|
|O0000004| livrée|mobile_app|
|O0000005|annulée|       web|
+--------+-------+----------+
only showing top 5 rows

livrees : 38890
+---------+-----+
|   statut|count|
+---------+-----+
|retournée| 2484|
|   livrée|38890|
| en_cours| 4058|
|  annulée| 4568|
+---------+-----+

+----------+-----+
|     canal|count|
+----------+-----+
|mobile_app|32519|
|       web|17481|
+----------+-----+



Ellipsis

## Exercice C.3 — Spark vs Pandas : mesurez

Chronométrez Spark sur `orders.csv`, comparez à votre mesure Pandas du TP 1
(même fichier, échelle 0.1). Lequel gagne sur ce petit volume ?

In [17]:
import time

t0 = time.perf_counter()

n = (
    spark.read
    .option("header", True)
    .csv("../data/orders.csv")
    .count()
)

t_spark = time.perf_counter() - t0

print("Spark : %.2f s pour %d lignes" % (t_spark, n))

# Temps obtenu avec Pandas dans le TP1
t_pandas = 0.21

print("Pandas (TP1) : %s s" % t_pandas)

Spark : 0.34 s pour 50000 lignes
Pandas (TP1) : 0.21 s


Sur ce petit volume, Pandas est plus rapide que Spark. Spark devient surtout avantageux lorsque le volume de données est important et nécessite un traitement distribué.

## Exercice D — Lire la Spark UI

Ouvrez http://localhost:4040. Cette cellule donne les partitions ; le reste
s'observe **dans le navigateur** (onglets Jobs, Stages, SQL/DataFrame).

In [20]:
# === À COMPLÉTER ===
print("Partitions orders :", orders.rdd.getNumPartitions())
print("Coeurs disponibles :", spark.sparkContext.defaultParallelism)

# Relancez le WordCount pour le retrouver dans l'onglet Jobs :
mots.orderBy(col("count").desc()).show(5)

Partitions orders : 1
Coeurs disponibles : 12
+-----+-----+
|  mot|count|
+-----+-----+
|  les|    5|
|  des|    5|
|   de|    4|
|spark|    4|
|   et|    4|
+-----+-----+
only showing top 5 rows



### D — Relevés (complétez en markdown)
- Nombre de stages du WordCount : plusieurs stages liés au shuffle
- Shuffle Read / Write du stage d'agrégation : Shuffle Write = 5,2 KiB
- Nombre de tasks par stage / nombre de partitions : 1 task observée / 1 partition
- Tasks en parallèle vs nombre de cœurs : 1 task pour 12 cœurs disponibles

## 5. Avant de pousser

Vérifiez : WordCount commenté ligne à ligne, observations Spark UI
renseignées, comparaison Spark/Pandas chiffrée. Puis :

```bash
git add notebooks/TP2_wordcount.ipynb
git commit -m "TP2 : WordCount PySpark, fil rouge, Spark UI"
git push
```

Pensez à **arrêter la session** en fin de travail : `spark.stop()`.

In [21]:
spark.stop()